# Embedding variance partitioning

How much of the structure in e5-large passage embeddings is explained by:
- **content annotations** (what happens in the passage)
- **form annotations** (how it's told)
- **genre tags** (from parent text)
- **period** (binned year)
- **abstractness score** (continuous)
- **language** (EN/FR)

Method: dbRDA-style variance partitioning. For each factor group, compute marginal R² (alone) and unique R² (after partialling out all other groups). Uses `build_feature_matrix` and `fit_partition_model` from `largeliterarymodels.analysis`.

In [ ]:
import clickhouse_connect
import numpy as np
import pandas as pd
from largeliterarymodels.analysis import build_feature_matrix
from abstraction.analysis import attach_extras, run_partition

CH = dict(host='localhost', port=8123, username='lltk', password='lltk')
c = clickhouse_connect.get_client(**CH)

## Pipeline

`attach_extras` merges genre tags, period dummies, abstractness, (optional) language, and passage embeddings onto a `build_feature_matrix` output. `run_partition` stacks the feature columns and calls `fit_partition_model`. Both live in `abstraction.analysis`.

## A) Form + Content overlap (English, 1,685 passages)

Uses form ensemble (derived/ensemble-maj4-trust60) ∩ content V3 (qwen3.5-35b-a3b). With `is_prose_fiction` filter → 1,310 passages.

In [ ]:
features, groups = build_feature_matrix(
    tasks=['passage-form', 'passage-content'],
    source_agents={'passage-content': 'qwen3.5-35b-a3b'},
    client=c,
)
print(f'Core features: {features.shape[1]} cols, {len(features)} passages')

feat, groups = attach_extras(features, groups, c, include_lang=False)
print(f'After extras: N={len(feat)}  |  ' + ', '.join(f'{k}({len(v)})' for k, v in groups.items()))

result = run_partition(feat, groups)
print(f"\nFull R²: {result.attrs['full_r2']:.4f}")
print(result.sort_values('unique_r2', ascending=False).to_string())

## B) Content-only, EN+FR (≈14,000 passages)

Drops form (English-only) to include French passages. Adds language as a factor.

In [ ]:
features_c, groups_c = build_feature_matrix(
    tasks=['passage-content'],
    source_agents={'passage-content': 'qwen3.5-35b-a3b'},
    client=c,
)
print(f'Core features: {features_c.shape[1]} cols, {len(features_c)} passages')

feat_c, groups_c = attach_extras(features_c, groups_c, c, include_lang=True)
print(f'After extras: N={len(feat_c)}  |  ' + ', '.join(f'{k}({len(v)})' for k, v in groups_c.items()))

result_c = run_partition(feat_c, groups_c)
print(f"\nFull R²: {result_c.attrs['full_r2']:.4f}")
print(result_c.sort_values('unique_r2', ascending=False).to_string())

## C) Content-only, English subset (≈10,900 passages)

Same as B but restricted to English — direct comparison with (A) to see whether excluding French changes the story.

In [ ]:
feat_en = feat_c[feat_c['lang'] == 'en'].copy()
groups_en = {k: v for k, v in groups_c.items() if k != 'language'}
print(f'English-only content: N={len(feat_en)}')

result_en = run_partition(feat_en, groups_en)
print(f"\nFull R²: {result_en.attrs['full_r2']:.4f}")
print(result_en.sort_values('unique_r2', ascending=False).to_string())

## D) Content-only, French subset

Same as C but restricted to French. Reveals within-French embedding structure — since French content annotations just landed, this is the first look at what organizes embedding space inside French fiction specifically.

In [ ]:
feat_fr = feat_c[feat_c['lang'] == 'fr'].copy()
groups_fr = {k: v for k, v in groups_c.items() if k != 'language'}
print(f'French-only content: N={len(feat_fr)}')

result_fr = run_partition(feat_fr, groups_fr)
print(f"\nFull R²: {result_fr.attrs['full_r2']:.4f}")
print(result_fr.sort_values('unique_r2', ascending=False).to_string())

## E) Top individual features

Per-feature marginal R² — which single annotations best predict embedding structure?

In [ ]:
from abstraction.analysis import per_feature_r2

print('Top 30 features in overlap set (A):')
print(per_feature_r2(feat, groups, top_n=30).to_string(index=False))

In [ ]:
print('Top 30 features in EN+FR content-only set (B):')
print(per_feature_r2(feat_c, groups_c, top_n=30).to_string(index=False))

In [ ]:
print('Top 30 features in French-only set (D):')
print(per_feature_r2(feat_fr, groups_fr, top_n=30).to_string(index=False))

In [ ]:
c.close()